In [ ]:
import os
from importlib import resources as impresources

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt, animation
from matplotlib.lines import Line2D

import mcfacts.vis.LISA as li
from mcfacts.fiducial_plots import make_gen_masks
from mcfacts.objects.snapshot import TxtSnapshotHandler
from mcfacts.vis import data
from mcfacts.vis import plotting
from mcfacts.vis import styles

In [ ]:
plt.style.use("mcfacts.vis.mcfacts_figures_dark")

figsize = "apj_col"

snapshot_handler = TxtSnapshotHandler()
settings = snapshot_handler.load_settings("", "../../runs/settings.txt")

file_path = "../../runs"
plots_dir = "../../runs/plots"

population_cabinet = snapshot_handler.load_cabinet(file_path, "population")

mergers = population_cabinet["blackholes_merged"]
lvk = population_cabinet["blackholes_lvk"]
emri = population_cabinet["blackholes_emri"]

mass_1 = mergers["mass"]
mass_2 = mergers["mass_2"]
chi_eff = mergers["chi_eff"]
mass_final = mergers["mass_final"]
orb_a = mergers["orb_a"]
chi_p = mergers["chi_p"]
time_merged = mergers["time_merged"]
v_kick = mergers["v_kick"]
spin_final = mergers["spin_final"]

merger_masks = (make_gen_masks(mergers["gen"], mergers["gen_2"])) # Man, I hate python

In [ ]:
def interactions_to_merge(
    settings,
    figsize,
    save_dir,
    lvk,
    *,
    merger_flag=-2,
    rel_tol=1e-6,
    require_merger=True,
    apply_stalling_cut=False,
):
    uid_all  = np.asarray(lvk["unique_id"])
    sep_all  = np.asarray(lvk["bin_sep"], dtype=float)
    flag_all = np.asarray(lvk["flag_merging"])

    if "time" in lvk:
        time_all = np.asarray(lvk["time"], dtype=float)
    else:
        time_all = None  # fall back to existing row order

    hardening_counts, softening_counts, total_counts = [], [], []

    for uid in np.unique(uid_all):
        mask = uid_all == uid
        sep  = sep_all[mask]
        flag = flag_all[mask]

        # --- enforce chronological order -------------------------------------
        if time_all is not None:
            order = np.argsort(time_all[mask], kind="stable")
            sep, flag = sep[order], flag[order]

        # --- locate the merger and truncate the history ----------------------
        merged_idx = np.flatnonzero(flag == merger_flag)
        if merged_idx.size == 0:
            if require_merger:
                continue
            end = sep.size
        else:
            end = merged_idx[0] + 1          # include the merger record;
                                             # use merged_idx[0] to exclude the
                                             # final inspiral transition instead
        sep = sep[:end]

        # --- optional stalling cut (see caveat in notes) ---------------------
        if apply_stalling_cut and hasattr(settings, "stalling_separation"):
            sep = sep[sep <= settings.stalling_separation]

        if sep.size < 2:                     # no transition can be defined
            hardening_counts.append(0)
            softening_counts.append(0)
            total_counts.append(0)
            continue

        # --- detect encounters via relative change in separation -------------
        prev, curr = sep[:-1], sep[1:]
        rel_change = (curr - prev) / np.abs(prev)
        changed    = np.abs(rel_change) > rel_tol

        hardening = int(np.count_nonzero(changed & (rel_change < 0)))
        softening = int(np.count_nonzero(changed & (rel_change > 0)))

        hardening_counts.append(hardening)
        softening_counts.append(softening)
        total_counts.append(hardening + softening)

    hardening_counts = np.asarray(hardening_counts)
    softening_counts = np.asarray(softening_counts)
    total_counts     = np.asarray(total_counts)
    n = total_counts.size

    stats = {
        "n_binaries":          n,
        "mean_interactions":   total_counts.mean()            if n     else np.nan,
        "std_interactions":    total_counts.std(ddof=1)       if n > 1 else np.nan,
        "sem_interactions":    total_counts.std(ddof=1)/np.sqrt(n) if n > 1 else np.nan,
        "median_interactions": float(np.median(total_counts)) if n     else np.nan,
        "mean_hardening":      hardening_counts.mean()        if n     else np.nan,
        "mean_softening":      softening_counts.mean()        if n     else np.nan,
    }

    # --- plot ----------------------------------------------------------------
    os.makedirs(save_dir, exist_ok=True)
    fig, ax = plt.subplots(figsize=(5,6))

    ax.set_xlim((-0.5,5.5))

    if n and total_counts.max() > 0:
        edges = np.arange(0, total_counts.max() + 2) - 0.5   # integer-centered
        ax.hist(hardening_counts, bins=edges, color="b", alpha=0.5,
                label="Hardening encounters")
        ax.hist(softening_counts, bins=edges, color="r", alpha=0.5,
                label="Softening encounters")
        ax.axvline(stats["mean_hardening"], color="white", ls="--",
                   label=f"Mean hardening events = {stats['mean_hardening']:.2f}")
    ax.set_xlabel("# of dynamical encounters")
    ax.set_ylabel("# of binaries")
    ax.legend()
    fig.savefig(os.path.join(save_dir, "interactions_to_merge.svg"), format="svg")

    plt.show(fig)

    print(f"{n} merging binaries | mean interactions to merge = "
          f"{stats['mean_interactions']:.3f} ± {stats['sem_interactions']:.3f} (SEM)")

interactions_to_merge(settings, figsize, plots_dir, lvk)

In [ ]:
def animate_strain_vs_freq(
    lvk,
    save_dir,
    figsize      = "apj_col",
    sim_tmax     = 700000,     # years
    dt_yr        = 10000,      # years per snapshot
    n_interp     = 15,         # interpolation frames between snapshots
    fps          = 30,
    dpi          = 300,        # render resolution
    crf          = 9,          # x264 quality (18 ~ visually lossless)
    fade_seconds = 5.0,        # LISA fade-out for binaries that DON'T merge (ionized etc.)
    fade_in_seconds = 2,       # ramp-in for newly appearing points (reduces "jumpiness")
):
    def gen_key(g1_val, g2_val):
        if g1_val == 1 and g2_val == 1:
            return "g1"
        if (g1_val == 2 and g2_val in (1, 2)) or (g1_val == 1 and g2_val == 2):
            return "g2"
        return "gX"

    fade_out = max(int(round(fade_seconds * fps)), 1)
    fade_in = max(int(round(fade_in_seconds * fps)), 1)

    lvk = pd.DataFrame(lvk)

    # LIGO O3 ASD files
    H1 = impresources.files(data) / "O3-H1-C01_CLEAN_SUB60HZ-1262197260.0_sensitivity_strain_asd.txt"
    L1 = impresources.files(data) / "O3-L1-C01_CLEAN_SUB60HZ-1240573680.0_sensitivity_strain_asd.txt"
    dfh1 = pd.read_csv(H1, sep="\t", header=None)
    dfl1 = pd.read_csv(L1, sep="\t", header=None)

    # LISA curve via the LISA_Sensitivity LISA.py module
    lisa = li.LISA()
    lisa_freq = np.logspace(np.log10(1.0e-5), np.log10(1.0e0), 1000)
    lisa_sn = lisa.Sn(lisa_freq)

    fig, axs = plt.subplots(1, 2, figsize=(plotting.set_size(figsize)[0] * 2, 2.9))
    fig.subplots_adjust(wspace=0.05)
    lisa_ax, lvk_ax = axs

    # Static LISA elements
    lisa_ax.set_xlim(0.5e-7, 1.0e1)
    lisa_ax.set_ylim(1.0e-26, 1.0e-15)
    lisa_ax.set_xscale("log")
    lisa_ax.set_yscale("log")
    lisa_ax.loglog(
        lisa_freq, np.sqrt(lisa_freq * lisa_sn),
        label="LISA Sensitivity", color="tab:orange", linewidth=1, zorder=0,
    )
    lisa_ax.set_xlabel(r"$\nu_{\rm GW}$ [Hz]")
    lisa_ax.set_ylabel(r"$h_{\rm char}$")

    # Static LVK elements
    lvk_ax.set_xlim(5e0, 1e4)
    lvk_ax.set_ylim(2.0e-24, 1.0e-19)
    lvk_ax.set_xscale("log")
    lvk_ax.set_yscale("log")
    lvk_ax.loglog(dfh1[0], dfh1[1], label="LIGO O3, H1 Sens.", color="tab:blue",   linewidth=1, zorder=0)
    lvk_ax.loglog(dfl1[0], dfl1[1], label="LIGO O3, L1 Sens.", color="tab:orange", linewidth=1, zorder=0)
    lvk_ax.set_xlabel(r"$\nu_{\rm GW}$ [Hz]")
    lvk_ax.yaxis.tick_right()
    lvk_ax.yaxis.set_label_position("right")
    lvk_ax.set_ylabel(r"$h_{\rm 0}$", rotation=-90, labelpad=15)

    # Time label
    time_text = lisa_ax.text(
        0.03, 0.97, "", transform=lisa_ax.transAxes,
        ha="left", va="top", fontsize=8,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7),
    )

    snapshot_times = np.arange(0, sim_tmax + dt_yr, dt_yr)
    snapshot_num = len(snapshot_times)

    # Separate out the lvk rows from the lisa rows
    lisa_dataframe = lvk[lvk["flag_merging"] != -2]
    lvk_dataframe = lvk[lvk["flag_merging"] == -2]

    # LISA gw tracks, one per binary, mapped onto the snapshot grid
    lisa_meta = {}
    for uuid, rows in lisa_dataframe.groupby("unique_id"):
        g_idx = np.rint(rows["time"].values / dt_yr).astype(int)
        by_snap = {int(gi): i for i, gi in enumerate(g_idx)}
        last_row = rows.iloc[by_snap[int(g_idx.max())]]

        lisa_meta[uuid] = {
            "rows": rows.sort_values("time").reset_index(drop=True),
            "by_snap": by_snap,
            "snap_first": int(g_idx.min()),
            "snap_last": int(g_idx.max()),
            "gk_last": gen_key(last_row["gen"], last_row["gen_2"]),
        }

    # LVK merger events appear at merger time, then persist forever
    lvk_meta = []
    for _, r in lvk_dataframe.iterrows():
        lvk_meta.append({
            "appear_frame": int(round(r["time"] / dt_yr)) * n_interp,
            "x": float(r["gw_freq"]),
            "y": float(r["gw_strain"]),
            "gk": gen_key(r["gen"], r["gen_2"]),
        })

    total_frames = (snapshot_num - 1) * n_interp + 1
    print(f"Pre-computing {total_frames} frames (fade-out {fade_seconds:g}s, fade-in {fade_in_seconds:g}s ...")

    frames_data = []
    for i in range(snapshot_num):
        t_now = snapshot_times[i]
        n_steps = n_interp if i < snapshot_num - 1 else 1

        for interp_i in range(n_steps):
            current_frame = i * n_interp + interp_i

            #  LISA
            lisa_pts = []
            for bid, meta in lisa_meta.items():
                by_snap = meta["by_snap"]
                snap_first = meta["snap_first"]
                snap_last = meta["snap_last"]

                birth_frame = snap_first * n_interp
                last_live_frame = snap_last * n_interp

                # Not tracking yet
                if current_frame < birth_frame:
                    continue

                # Tracking
                if current_frame <= last_live_frame:
                    if i not in by_snap:
                        continue

                    rows = meta["rows"]
                    row_now = rows.iloc[by_snap[i]]

                    if (i + 1) in by_snap and interp_i > 0:
                        row_next = rows.iloc[by_snap[i + 1]]
                        t = interp_i / n_interp

                        x = 10 ** (np.log10(row_now["gw_freq"])
                                   + t * (np.log10(row_next["gw_freq"])
                                          - np.log10(row_now["gw_freq"])))

                        y = 10 ** (np.log10(row_now["gw_char_strain"])
                                   + t * (np.log10(row_next["gw_char_strain"])
                                          - np.log10(row_now["gw_char_strain"])))
                    else:
                        x = row_now["gw_freq"]
                        y = row_now["gw_char_strain"]

                    gk = gen_key(row_now["gen"], row_now["gen_2"])
                    alpha = min((current_frame - birth_frame) / fade_in, 1.0)

                else:
                    # Number of frames since last data point
                    dt_end = current_frame - last_live_frame

                    # Ionized / never merged: fade out in place.
                    if dt_end >= fade_out:
                        continue

                    rows = meta["rows"]
                    row_last = rows.iloc[by_snap[snap_last]]
                    x = row_last["gw_freq"]
                    y = row_last["gw_char_strain"]
                    alpha = 1.0 - dt_end / fade_out
                    gk = meta["gk_last"]

                if alpha > 0.0:
                    lisa_pts.append({"x": x, "y": y, "alpha": alpha, "gk": gk})

            # LVK
            lvk_pts = []
            for ev in lvk_meta:
                if ev["appear_frame"] > current_frame:
                    continue
                alpha = min((current_frame - ev["appear_frame"]) / fade_in, 1.0)
                lvk_pts.append({"x": ev["x"], "y": ev["y"], "alpha": alpha, "gk": ev["gk"]})

            frames_data.append({
                "lisa_pts": lisa_pts,
                "lvk_pts": lvk_pts,
                "t_now": t_now,
                "tau": interp_i / n_interp,
            })

        if i % 10 == 0:
            print(f"  snapshot {i}/{snapshot_num - 1}")

    print(f"Done. {len(frames_data)} frames total.")

    scatter_artists = {}
    for gk, gs in styles.gen_styles.items():
        s_lisa = lisa_ax.scatter([], [], s=gs["size"], marker=gs["marker"], edgecolors=gs["color"], facecolors="none", zorder=2)
        s_lvk = lvk_ax.scatter([], [], s=gs["size"], marker=gs["marker"], edgecolors=gs["color"], facecolors="none", zorder=2)

        scatter_artists[gk] = {"lisa": s_lisa, "lvk": s_lvk}

    gen_handles = []
    for gk in styles.gen_styles:
        Line2D([], [], linestyle="none", marker=styles.gen_styles[gk]["marker"], markersize=5,
               markeredgecolor=styles.gen_styles[gk]["color"], markerfacecolor="none",
               label=styles.gen_styles[gk]["label"])

    # LISA sensitivity / legend
    lisa_handles = [lisa_ax.lines[0]] + gen_handles

    if figsize == "apj_col":
        lisa_ax.legend(handles=lisa_handles, fontsize=7, loc="upper right")
        lvk_ax.legend(fontsize=7, loc="upper left")
    else:
        lisa_ax.legend(handles=lisa_handles, loc="upper right")
        lvk_ax.legend(loc="upper left")

    # ugh, ugly nested methods, why must this be the way python animations work...
    def update(frame_idx):
        frame_data = frames_data[frame_idx]

        def fill(scatter_key, pts, with_alpha):
            fill_artists = []
            buckets = {gk: {"x": [], "y": [], "a": []} for gk in styles.gen_styles}

            for p in pts:
                bucket = buckets[p["gk"]]
                bucket["x"].append(p["x"])
                bucket["y"].append(p["y"])
                bucket["a"].append(p["alpha"] if with_alpha else 1.0)

            for gen_key in styles.gen_styles:
                bucket = buckets[gen_key]
                n = len(bucket["x"])
                scatter_artist = scatter_artists[gen_key][scatter_key]

                if n:
                    rgba = np.zeros((n, 4))
                    rgba[:, :3] = styles.gen_styles[gen_key]["rgb"]
                    rgba[:, 3] = styles.gen_styles[gen_key]["alpha"] * np.asarray(bucket["a"])
                    scatter_artist.set_offsets(np.column_stack([bucket["x"], bucket["y"]]))
                    scatter_artist.set_edgecolors(rgba)
                else:
                    scatter_artist.set_offsets(np.empty((0, 2)))
                    scatter_artist.set_edgecolors(np.empty((0, 4)))

                fill_artists.append(scatter_artist)

            return fill_artists

        artists = []
        artists += fill("lisa", frame_data["lisa_pts"], with_alpha=True)
        artists += fill("lvk", frame_data["lvk_pts"], with_alpha=True)

        time_text.set_text(f"t = {(frame_data["t_now"] + frame_data["tau"] * dt_yr) / 1e6:.2f} Myr")
        artists.append(time_text)

        return artists

    ani = animation.FuncAnimation(
        fig, update, frames=len(frames_data),
        interval=1000 / fps, blit=True,
    )

    mp4_path = os.path.join(save_dir, "gw_strain.mp4")

    print(f"Saving MP4: {mp4_path} (dpi={dpi}, crf={crf})")

    writer_mp4 = animation.FFMpegWriter(
        fps=fps, codec="libx264",
        extra_args=["-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2",
                    "-crf", str(crf), "-pix_fmt", "yuv420p"],
    )

    ani.save(mp4_path, writer=writer_mp4, dpi=dpi)

    print("MP4 saved.")

    plt.close(fig)

animate_strain_vs_freq(
    lvk         = lvk,
    save_dir    = plots_dir,
    sim_tmax    = settings.active_timestep_num * settings.active_timestep_duration_yr,
    dt_yr       = settings.active_timestep_duration_yr
)